# COMP 3610 Assignment 3

## LLM Applications and Distributed Computing


## Setup

In [1]:
import os
import json
import time
import shutil
import urllib.request
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from openai import OpenAI
from pypdf import PdfReader

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


KeyboardInterrupt: 

## Configuration


In [ ]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data'
DOCS_DIR = BASE_DIR / 'docs'
OUTPUT_DIR = BASE_DIR / 'output'
CHROMA_DIR = BASE_DIR / 'chroma_db'

for folder in [DATA_DIR, DOCS_DIR, OUTPUT_DIR]:
    folder.mkdir(exist_ok=True)

TAXI_URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet'
TAXI_PATH = DATA_DIR / 'yellow_tripdata_2024-01.parquet'

LLM_BASE_URL = 'https://synapse.sergiomathurin.com/v1'
LLM_API_KEY = ''
DEFAULT_MODEL = 'llama3.3-70b-instruct'

DOC_URLS = {
    '01_tlc_annual_report_2025.pdf': 'https://www.nyc.gov/assets/tlc/downloads/pdf/annual_report_2025.pdf',
    '02_tlc_annual_report_2024.pdf': 'https://www.nyc.gov/assets/tlc/downloads/pdf/annual_report_2024.pdf',
    '03_tlc_driver_rules.pdf': 'https://www.nyc.gov/assets/tlc/downloads/pdf/drivrules.pdf',
    '04_tlc_rule_book_chapter_80.pdf': 'https://www.nyc.gov/assets/tlc/downloads/pdf/rule_book_current_chapter_80.pdf',
    '05_nyc_streets_plan_2021.pdf': 'https://www.nyc.gov/html/dot/downloads/pdf/nyc-streets-plan.pdf',
    '06_nyc_streets_plan_update_2026.pdf': 'https://www.nyc.gov/html/dot/downloads/pdf/nyc-streets-plan-update.pdf',
    '07_nyc_mobility_report_2019.pdf': 'https://www.nyc.gov/html/dot/downloads/pdf/mobility-report-singlepage-2019.pdf',
}


## Data and document download


In [ ]:
if not TAXI_PATH.exists():
    urllib.request.urlretrieve(TAXI_URL, TAXI_PATH)

for name, url in DOC_URLS.items():
    target = DOCS_DIR / name
    if not target.exists():
        urllib.request.urlretrieve(url, target)

print('Taxi file:', TAXI_PATH)
print('PDF count:', len(list(DOCS_DIR.glob('*.pdf'))))


## Part 1 - Distributed Data Processing with Spark


### Task 1.1 - Spark environment setup and data loading


In [ ]:
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('COMP3610_Assignment3') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

print('Spark version:', spark.version)
print('App name:', spark.sparkContext.appName)
print('Master:', spark.sparkContext.master)
print('Default parallelism:', spark.sparkContext.defaultParallelism)


In [ ]:
start = time.time()
df_spark = spark.read.parquet(str(TAXI_PATH))
spark_load_time = time.time() - start

start = time.time()
spark_row_count = df_spark.count()
spark_action_time = time.time() - start

start = time.time()
df_pandas = pd.read_parquet(TAXI_PATH)
pandas_load_time = time.time() - start

print('Spark metadata load time:', round(spark_load_time, 3), 's')
print('Spark count time:', round(spark_action_time, 3), 's')
print('Pandas load time:', round(pandas_load_time, 3), 's')
print('Row count:', f'{spark_row_count:,}')
print('Partition count:', df_spark.rdd.getNumPartitions())

df_spark.printSchema()
df_spark.show(5, truncate=False)


### Task 1.2 - Data cleaning and feature engineering


In [ ]:
critical_cols = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'fare_amount',
    'trip_distance',
    'tip_amount'
]

initial_count = df_spark.count()
non_null_df = df_spark.dropna(subset=critical_cols)
after_null_count = non_null_df.count()

valid_df = non_null_df.filter(F.col('trip_distance') > 0)
valid_df = valid_df.filter(F.col('fare_amount') >= 0)
valid_df = valid_df.filter(F.col('fare_amount') <= 500)
valid_df = valid_df.filter(F.col('tpep_dropoff_datetime') >= F.col('tpep_pickup_datetime'))
after_valid_count = valid_df.count()

cleaned_df = valid_df.withColumn(
    'trip_duration_minutes',
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 60.0
).withColumn(
    'trip_speed_mph',
    F.when(
        F.col('trip_duration_minutes') > 0,
        F.col('trip_distance') / (F.col('trip_duration_minutes') / 60.0)
    ).otherwise(F.lit(None))
).withColumn(
    'pickup_hour',
    F.hour('tpep_pickup_datetime')
).withColumn(
    'pickup_day_of_week',
    F.dayofweek('tpep_pickup_datetime')
).withColumn(
    'tip_percentage',
    F.when(
        F.col('fare_amount') > 0,
        (F.col('tip_amount') / F.col('fare_amount')) * 100
    ).otherwise(F.lit(None))
)

final_count = cleaned_df.count()

print('Initial rows:', f'{initial_count:,}')
print('Removed by null filtering:', f'{initial_count - after_null_count:,}')
print('Removed by validity filtering:', f'{after_null_count - after_valid_count:,}')
print('Final cleaned rows:', f'{final_count:,}')

cleaned_df.select(
    'trip_distance', 'fare_amount', 'tip_amount',
    'trip_duration_minutes', 'trip_speed_mph',
    'pickup_hour', 'pickup_day_of_week', 'tip_percentage'
).show(5, truncate=False)


Null filtering and validity filtering are separated so the row removal at each stage is explicit. Derived columns are created with Spark functions only.


### Task 1.3 - Spark SQL analytics


In [ ]:
cleaned_df.createOrReplaceTempView('taxi_trips')


In [ ]:
query1 = """
SELECT pickup_hour,
       COUNT(*) AS trip_count,
       ROUND(AVG(fare_amount), 2) AS avg_fare,
       ROUND(AVG(tip_percentage), 2) AS avg_tip_pct
FROM taxi_trips
GROUP BY pickup_hour
ORDER BY trip_count DESC
LIMIT 10
"""

query2 = """
SELECT pickup_day_of_week,
       ROUND(AVG(trip_speed_mph), 2) AS avg_speed_mph,
       ROUND(AVG(trip_distance), 2) AS avg_distance,
       ROUND(AVG(trip_duration_minutes), 2) AS avg_duration_min
FROM taxi_trips
WHERE trip_speed_mph IS NOT NULL
GROUP BY pickup_day_of_week
ORDER BY avg_speed_mph DESC
"""

query3 = """
WITH revenue_by_day_loc AS (
    SELECT pickup_day_of_week,
           PULocationID,
           ROUND(SUM(total_amount), 2) AS total_revenue
    FROM taxi_trips
    GROUP BY pickup_day_of_week, PULocationID
), ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY pickup_day_of_week
               ORDER BY total_revenue DESC
           ) AS revenue_rank
    FROM revenue_by_day_loc
)
SELECT pickup_day_of_week, PULocationID, total_revenue, revenue_rank
FROM ranked
WHERE revenue_rank <= 5
ORDER BY pickup_day_of_week, revenue_rank
"""

query4 = """
WITH hourly_counts AS (
    SELECT pickup_hour,
           COUNT(*) AS trip_count
    FROM taxi_trips
    GROUP BY pickup_hour
), running AS (
    SELECT pickup_hour,
           trip_count,
           SUM(trip_count) OVER (ORDER BY pickup_hour) AS cumulative_trips,
           SUM(trip_count) OVER () AS total_trips
    FROM hourly_counts
)
SELECT pickup_hour,
       trip_count,
       cumulative_trips,
       ROUND(cumulative_trips * 100.0 / total_trips, 2) AS cumulative_pct
FROM running
ORDER BY pickup_hour
"""

query5 = """
SELECT CASE
           WHEN trip_distance < 2 THEN 'short'
           WHEN trip_distance <= 10 THEN 'medium'
           ELSE 'long'
       END AS trip_category,
       ROUND(AVG(fare_amount), 2) AS avg_fare,
       ROUND(AVG(trip_distance), 2) AS avg_distance,
       ROUND(AVG(tip_percentage), 2) AS avg_tip_pct
FROM taxi_trips
GROUP BY CASE
           WHEN trip_distance < 2 THEN 'short'
           WHEN trip_distance <= 10 THEN 'medium'
           ELSE 'long'
       END
ORDER BY avg_distance
"""

for i, sql_text in enumerate([query1, query2, query3, query4, query5], start=1):
    print('=' * 80)
    print('Query', i)
    spark.sql(sql_text).show(30, truncate=False)


Query 1 isolates high-volume hours and compares fare and tip behavior across those hours.

Query 2 compares average speed across days and includes distance and duration so the speed result has context.

Query 3 uses a window function to rank pickup locations by revenue within each day.

Query 4 identifies the point where daily trip volume passes the midpoint.

Query 5 compares tip percentage across trip-length groups instead of only comparing fare levels.


### Task 1.4 - Performance optimization


In [ ]:
repeated_sql = """
SELECT pickup_hour, COUNT(*) AS trip_count, ROUND(AVG(fare_amount), 2) AS avg_fare
FROM taxi_trips
GROUP BY pickup_hour
ORDER BY pickup_hour
"""

start = time.time()
spark.sql(repeated_sql).count()
uncached_time = time.time() - start

cleaned_df.cache()
cleaned_df.count()
cleaned_df.createOrReplaceTempView('taxi_trips_cached')

start = time.time()
spark.sql(repeated_sql.replace('taxi_trips', 'taxi_trips_cached')).count()
cached_time = time.time() - start

print('Uncached query time:', round(uncached_time, 3), 's')
print('Cached query time:', round(cached_time, 3), 's')


In [ ]:
partition_output = OUTPUT_DIR / 'cleaned_by_hour'
if partition_output.exists():
    shutil.rmtree(partition_output)

cleaned_df.write.mode('overwrite').partitionBy('pickup_hour').parquet(str(partition_output))
partitioned_df = spark.read.parquet(str(partition_output))
hour17_df = partitioned_df.filter(F.col('pickup_hour') == 17)

hour17_df.explain(mode='formatted')
print('Hour 17 rows:', hour17_df.count())


In [ ]:
explain_sql = """
SELECT pickup_hour, COUNT(*) AS trip_count, ROUND(AVG(total_amount), 2) AS avg_total
FROM taxi_trips
GROUP BY pickup_hour
ORDER BY trip_count DESC
"""

spark.sql(explain_sql).explain(mode='formatted')


Cache timing measures repeated access cost. Partitioned Parquet supports partition pruning on `pickup_hour`. The physical plan is expected to include scan, filter, exchange, and aggregate operators.


## Part 2 - Retrieval-Augmented Generation


### Task 2.1 - Document ingestion and chunking


In [ ]:
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

raw_documents = []
for pdf_path in sorted(DOCS_DIR.glob('*.pdf')):
    reader = PdfReader(str(pdf_path))
    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text()
        if text and text.strip():
            raw_documents.append(
                Document(
                    page_content=text,
                    metadata={'source': pdf_path.name, 'page': page_num}
                )
            )

print('Document pages loaded:', len(raw_documents))
print('Distinct PDFs:', len({doc.metadata['source'] for doc in raw_documents}))
print(raw_documents[0].metadata)
print(raw_documents[0].page_content[:600])


In [ ]:
def make_chunks(documents, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.split_documents(documents)

chunks_500 = make_chunks(raw_documents, 500, 100)
chunks_1000 = make_chunks(raw_documents, 1000, 200)

print('Chunks (500/100):', len(chunks_500))
print('Chunks (1000/200):', len(chunks_1000))
print(chunks_500[0].metadata)
print(chunks_500[0].page_content[:400])


Chunking uses overlap so content near boundaries is not dropped. Two chunk sizes are kept for later comparison.


### Task 2.2 - Embeddings and vector database


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

vectorstore_500 = Chroma.from_documents(
    documents=chunks_500,
    embedding=embedding_model,
    persist_directory=str(CHROMA_DIR / 'chunks_500'),
    collection_name='assignment3_chunks_500'
)

vectorstore_1000 = Chroma.from_documents(
    documents=chunks_1000,
    embedding=embedding_model,
    persist_directory=str(CHROMA_DIR / 'chunks_1000'),
    collection_name='assignment3_chunks_1000'
)

sample_results = vectorstore_500.similarity_search_with_score(
    'What rules apply to TLC drivers?',
    k=3
)
for doc, score in sample_results:
    print(round(score, 4), doc.metadata)
    print(doc.page_content[:250])
    print('-' * 60)


### Task 2.3 - RAG pipeline


In [ ]:
client = None
if LLM_API_KEY != '' and LLM_API_KEY.strip():       #Include the API key
    client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)


def call_llm(messages, model=DEFAULT_MODEL, temperature=0, max_tokens=500):
    if client is None:
        raise ValueError('Invalid API key.')
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content


def format_sources(docs):
    seen = []
    for doc in docs:
        ref = (doc.metadata.get('source'), doc.metadata.get('page'))
        if ref not in seen:
            seen.append(ref)
    return seen


def ask_rag(question, vectorstore, k=4):
    docs = vectorstore.similarity_search(question, k=k)
    context = '\n\n'.join([
        f"[{i+1}] Source: {doc.metadata.get('source')} | Page: {doc.metadata.get('page')}\n{doc.page_content}"
        for i, doc in enumerate(docs)
    ])
    prompt = [
        {
            'role': 'system',
            'content': (
                'Answer only from the supplied context. '
                'If the context is insufficient, state that explicitly. '
                'Cite sources in the form (source, page).'
            )
        },
        {
            'role': 'user',
            'content': f'Question: {question}\n\nContext:\n{context}'
        }
    ]
    answer = call_llm(prompt)
    return answer, docs


In [ ]:
rag_test_questions = [
    'What topics are covered in the TLC 2025 annual report?',
    'What are the driver conduct requirements in the TLC driver rules?',
    'What goals are stated in the NYC Streets Plan?',
    'What transportation indicators are discussed in the mobility report?',
    'What restrictions appear in the TLC rule book chapter 80?'
]

if client is not None:
    rag_results = []
    for question in rag_test_questions:
        answer, docs = ask_rag(question, vectorstore_500, k=4)
        refs = format_sources(docs)
        rag_results.append({'question': question, 'answer': answer, 'sources': refs})
        print('QUESTION:', question)
        print('ANSWER:', answer)
        print('SOURCES:', refs)
        print('=' * 100)
else:
    print('LLM sections are ready. Insert API key to run this cell.')


### Task 2.4 - RAG evaluation and analysis


In [ ]:
eval_cases = [
    {'question': 'Which report discusses wheelchair accessible vehicles?', 'expected_source_contains': 'annual_report_2025'},
    {'question': 'Which document contains detailed driver conduct rules?', 'expected_source_contains': 'driver_rules'},
    {'question': 'Which document states city street design goals?', 'expected_source_contains': 'streets_plan'},
    {'question': 'Which document reports transportation indicators and trends?', 'expected_source_contains': 'mobility_report'},
    {'question': 'Which document describes TLC chapter 80 requirements?', 'expected_source_contains': 'chapter_80'},
    {'question': 'Which annual report references driver pay or platform protections?', 'expected_source_contains': 'annual_report_2025'},
    {'question': 'Which document addresses safer and more sustainable streets?', 'expected_source_contains': 'streets_plan'},
    {'question': 'Which document is focused on taxi driver rules rather than planning policy?', 'expected_source_contains': 'driver_rules'},
    {'question': 'Which report summarizes mobility conditions across the city?', 'expected_source_contains': 'mobility_report'},
    {'question': 'Which document is a TLC rule book chapter rather than an annual report?', 'expected_source_contains': 'chapter_80'}
]

retrieval_rows = []
for case in eval_cases:
    docs = vectorstore_500.similarity_search(case['question'], k=4)
    retrieved_sources = [doc.metadata['source'] for doc in docs]
    retrieval_hit = any(case['expected_source_contains'] in source for source in retrieved_sources)
    retrieval_rows.append({
        'question': case['question'],
        'expected_source_contains': case['expected_source_contains'],
        'retrieved_sources': retrieved_sources,
        'retrieval_hit': retrieval_hit
    })

retrieval_df = pd.DataFrame(retrieval_rows)
retrieval_accuracy = retrieval_df['retrieval_hit'].mean() * 100
print('Retrieval accuracy:', round(retrieval_accuracy, 2), '%')
retrieval_df


In [ ]:
if client is not None:
    answer_rows = []
    for case in eval_cases:
        answer, docs = ask_rag(case['question'], vectorstore_500, k=4)
        answer_rows.append({
            'question': case['question'],
            'answer': answer,
            'sources': format_sources(docs)
        })
    answer_df = pd.DataFrame(answer_rows)
    answer_df
else:
    print('Insert API key to run answer-quality evaluation.')


Retrieval failure: expected source not present in top-k results.

Generation failure: expected source retrieved, but final answer is incomplete, incorrect, or unsupported by the retrieved context.

Potential improvements: larger `k`, query rewriting, metadata-aware reranking, chunk-size tuning, stricter grounding prompt, and answer verification against retrieved text.


## Part 3 - Integrated analytics application


### Task 3.1 - Query router


In [ ]:
ROUTER_SYSTEM_PROMPT = '''
Classify the user question into one of three labels: DATA, DOCUMENT, HYBRID.
Return valid JSON with keys: category, reasoning.
Use DATA for questions answerable from the taxi dataset.
Use DOCUMENT for questions answerable from the PDF corpus.
Use HYBRID for questions requiring both.
If ambiguous, return HYBRID.
'''


def parse_json_text(text):
    text = text.strip()
    if text.startswith('```'):
        text = text.split('\n', 1)[1].rsplit('```', 1)[0]
    return json.loads(text)


def route_query(question):
    raw = call_llm([
        {'role': 'system', 'content': ROUTER_SYSTEM_PROMPT},
        {'role': 'user', 'content': question}
    ], temperature=0, max_tokens=150)
    return parse_json_text(raw)


In [ ]:
router_test_set = [
    ('What was the average fare on Mondays?', 'DATA'),
    ('What is the average tip percentage by pickup hour?', 'DATA'),
    ('Which pickup locations generated the most revenue?', 'DATA'),
    ('At what hour does cumulative trip volume pass 50 percent?', 'DATA'),
    ('Which trip category had the highest tip percentage?', 'DATA'),
    ('What rules apply to TLC drivers?', 'DOCUMENT'),
    ('What goals are stated in the NYC Streets Plan?', 'DOCUMENT'),
    ('What does the mobility report say about transportation conditions?', 'DOCUMENT'),
    ('What topics are covered in the TLC annual report?', 'DOCUMENT'),
    ('What appears in TLC chapter 80?', 'DOCUMENT'),
    ('How do observed trip patterns compare with policy goals in the Streets Plan?', 'HYBRID'),
    ('How do tipping patterns compare with TLC guidance for drivers?', 'HYBRID'),
    ('Do high-demand hours align with policy concerns about congestion?', 'HYBRID'),
    ('How does actual revenue concentration compare with transportation planning priorities?', 'HYBRID'),
    ('How do travel demand patterns relate to the mobility report findings?', 'HYBRID')
]

if client is not None:
    router_rows = []
    for question, expected in router_test_set:
        pred = route_query(question)
        router_rows.append({
            'question': question,
            'expected': expected,
            'predicted': pred['category'],
            'reasoning': pred['reasoning']
        })
    router_df = pd.DataFrame(router_rows)
    router_accuracy = (router_df['expected'] == router_df['predicted']).mean() * 100
    print('Router accuracy:', round(router_accuracy, 2), '%')
    router_df
else:
    print('Insert API key to run the router test set.')


### Task 3.2 - Data query handler


In [ ]:
SQL_SYSTEM_PROMPT = '''
Generate valid Spark SQL for the view named taxi_trips.
Return valid JSON with one key: sql.
Use only existing columns.
'''

ANSWER_SYSTEM_PROMPT = '''
Generate a concise technical answer from the SQL result shown by the user.
Do not invent values.
'''


def generate_sql(question, error_message=None):
    user_text = f'Question: {question}'
    if error_message:
        user_text += f'\nPrevious SQL error: {error_message}'
    raw = call_llm([
        {'role': 'system', 'content': SQL_SYSTEM_PROMPT},
        {'role': 'user', 'content': user_text}
    ], temperature=0, max_tokens=250)
    return parse_json_text(raw)['sql']


def answer_from_sql_result(question, result_rows):
    result_text = json.dumps(result_rows, indent=2, default=str)
    return call_llm([
        {'role': 'system', 'content': ANSWER_SYSTEM_PROMPT},
        {'role': 'user', 'content': f'Question: {question}\n\nSQL result:\n{result_text}'}
    ], temperature=0, max_tokens=250)


def handle_data_query(question):
    sql_text = generate_sql(question)
    try:
        result_df = spark.sql(sql_text)
    except Exception as e:
        sql_text = generate_sql(question, error_message=str(e))
        result_df = spark.sql(sql_text)
    rows = [row.asDict() for row in result_df.limit(20).collect()]
    final_answer = answer_from_sql_result(question, rows)
    return sql_text, rows, final_answer


In [ ]:
data_test_questions = [
    'What was the average fare on Mondays?',
    'Which pickup hour had the highest trip count?',
    'What is the average tip percentage for short trips?',
    'Which day had the highest average trip speed?',
    'What was the cumulative trip count by hour?'
]

if client is not None:
    for question in data_test_questions:
        sql_text, rows, final_answer = handle_data_query(question)
        print('QUESTION:', question)
        print('GENERATED SQL:\n', sql_text)
        print('RAW RESULTS:', rows)
        print('FINAL ANSWER:', final_answer)
        print('=' * 100)
else:
    print('Insert API key to run the data-query handler tests.')


### Task 3.3 - End-to-end demo


In [ ]:
def run_hybrid_query(question):
    data_sql, data_rows, data_answer = handle_data_query(question)
    rag_answer, rag_docs = ask_rag(question, vectorstore_500, k=4)
    combined_answer = call_llm([
        {
            'role': 'system',
            'content': 'Combine the structured analytics answer and the document-based answer into one concise technical response.'
        },
        {
            'role': 'user',
            'content': (
                f'Question: {question}\n\n'
                f'Data answer: {data_answer}\n\n'
                f'Document answer: {rag_answer}\n\n'
                f'Document sources: {format_sources(rag_docs)}'
            )
        }
    ], temperature=0, max_tokens=300)
    return {
        'sql': data_sql,
        'data_rows': data_rows,
        'data_answer': data_answer,
        'rag_answer': rag_answer,
        'rag_sources': format_sources(rag_docs),
        'combined_answer': combined_answer
    }


def run_system(question):
    route = route_query(question)
    category = route['category']
    if category == 'DATA':
        sql_text, rows, final_answer = handle_data_query(question)
        return {
            'category': category,
            'route_reasoning': route['reasoning'],
            'sql': sql_text,
            'rows': rows,
            'final_answer': final_answer
        }
    if category == 'DOCUMENT':
        answer, docs = ask_rag(question, vectorstore_500, k=4)
        return {
            'category': category,
            'route_reasoning': route['reasoning'],
            'sources': format_sources(docs),
            'final_answer': answer
        }
    hybrid = run_hybrid_query(question)
    hybrid['category'] = category
    hybrid['route_reasoning'] = route['reasoning']
    return hybrid


In [ ]:
end_to_end_queries = [
    'What was the average fare on Mondays?',
    'Which pickup hour had the highest trip count?',
    'What rules apply to TLC drivers?',
    'What goals are stated in the NYC Streets Plan?',
    'How do actual trip patterns compare with NYC transportation policy goals?',
    'How do tipping patterns compare with TLC and policy documents?'
]

if client is not None:
    demo_results = []
    for question in end_to_end_queries:
        result = run_system(question)
        demo_results.append({'question': question, 'result': result})
        print('QUESTION:', question)
        print(json.dumps(result, indent=2, default=str))
        print('=' * 100)
else:
    print('Insert API key to run the end-to-end demo.')


### Reflection

Structured analytics works well for questions that involve counts, averages, or trends since the taxi dataset can be handled directly with SQL. The document retrieval part works better for policy or regulation questions where the answers are in text rather than numbers. The hybrid approach helps when a question involves both, such as comparing taxi data with transport policies or explaining observed patterns using external documents.

Some issues can still happen in routing, SQL generation, and retrieval. For example, unclear questions might be sent to the wrong path, which leads to irrelevant results. SQL queries may fail if the prompt does not clearly match the dataset structure or expected columns. Retrieval can also miss useful information if the wording of the question is different from what is used in the documents. In some cases, the generated answers may be too general if the retrieved context is weak or not specific enough.

This could be improved by adding better handling for locations, improving how documents are retrieved, and testing more hybrid-type questions. On the Spark side, having clearer information about the dataset columns would help avoid invalid SQL. Overall, the system works, but the quality of results still depends on how well the queries and retrieved data align.

## Shutdown


In [ ]:
# spark.stop()
